# [drosophila_body_orientation_predictor](https://github.com/nehalsinghmangat/drosophila_body_orientation_predictor) — Data Pipeline

This notebook cleans, corrects, and augments the experimental data from [van Breugel et al. (2014)](https://www.sciencedirect.com/science/article/pii/S0960982213015820) to produce the fully preprocessed dataset used to train the heading predictor neural network.

The pipeline proceeds through the following stages, saving intermediate outputs to `../pipelinedata/`:

1. **`01_merged/`** — Raw HDF5 files merged across wind speeds (30/40/60 cm/s)
2. **`02_augmented/`** — Kinematic features added (groundspeed, airspeed, thrust, acceleration)
3. **`03_corrected/`** — 180° heading ambiguities resolved via naïve + convex-optimisation correction
4. **`04_filtered/`** — Short or uncorrectable trajectories removed
5. **`05_smoothed/`** — Residual jitter suppressed with Savitzky–Golay smoothing
6. **`06_final/`** — Output columns (`heading_angle_x`, `heading_angle_y`) added; ready for model training

Neural network training is handled in the companion notebook `model_training.ipynb`.

## Methods


### Data Preparation and Preprocessing

We use data from [van Breugel et al. (2014)](https://doi.org/10.1242/jeb.098665), collected across three constant-wind conditions (30, 40, 60 cm/s). Two data streams are merged per trajectory: (i) flight trajectory (x/y position and velocity) and (ii) body-orientation measurements captured as ellipse parameters. Trajectories shorter than a minimum duration are discarded to ensure adequate temporal context for model training.

Below, we load the raw HDF5 files for each wind speed, correct the airspeed reference frame, standardize field names, synchronize timestamps, and merge the two streams into a single CSV per wind speed.


In [ ]:
import os
import pandas as pd
import datetime

from utils import (
    correct_for_wind,
    remove_irrelevant_trajectory_data,
    remove_irrelevant_body_data,
    sync_time,
    join_all_body_and_trajec_df,
    transform_timestamps_to_start_at_zero,
)

windspeeds = ["30cms", "40cms", "60cms"]
for windspeed in windspeeds:
    trajec_df = pd.read_hdf(f"../experimentaldata/{windspeed}/flight_trajectories_3d_HCS_odor_horizon_matched.h5")
    body_df = pd.read_hdf(f"../experimentaldata/{windspeed}/body_orientations_HCS_odor_horizon_matched.h5")
    key_table = pd.read_hdf(f"../experimentaldata/{windspeed}/body_trajec_matches.h5")

    trajec_df = correct_for_wind(trajec_df)
    trajec_df = remove_irrelevant_trajectory_data(trajec_df)
    body_df = remove_irrelevant_body_data(body_df)

    synced_trajectory = sync_time(trajec_df)
    synced_body = sync_time(body_df)

    body_and_trajectory = join_all_body_and_trajec_df(synced_body, synced_trajectory, key_table)
    body_and_trajectory = [transform_timestamps_to_start_at_zero(t) for t in body_and_trajectory]
    body_and_trajectory = [t for t in body_and_trajectory if len(t) > 12]

    temp_fly_data = pd.concat(body_and_trajectory, ignore_index=True)
    os.makedirs(f"../pipelinedata/01_merged/{windspeed}", exist_ok=True)
    temp_fly_data.to_csv(f"../pipelinedata/01_merged/{windspeed}/fly_trajectories_with_body_orientations.csv", index=False)

temp_file_paths = [
    "../pipelinedata/01_merged/30cms/fly_trajectories_with_body_orientations.csv",
    "../pipelinedata/01_merged/40cms/fly_trajectories_with_body_orientations.csv",
    "../pipelinedata/01_merged/60cms/fly_trajectories_with_body_orientations.csv",
]
temp_dfs = [pd.read_csv(f) for f in temp_file_paths]
heading_and_trajectories = pd.concat(temp_dfs, ignore_index=True)
heading_and_trajectories.to_csv('../pipelinedata/01_merged/all_wind_heading_and_trajectories.csv', index=False)

#### Kinematic Feature Augmentation

We augment each merged trajectory with physically meaningful derived quantities: groundspeed, airspeed, linear acceleration, and thrust (magnitude and angle). These fields are required both for heading correction (thrust angle serves as a reference direction) and as inputs to the neural network. The heading angle is also initialized here from the ellipse short-axis angle (accurate only modulo π — corrected in the next step).


In [ ]:
import os
import pynumdiff as pynd
import numpy as np

from utils import augment_fly_trajectory

all_fly_data = pd.read_csv('../pipelinedata/01_merged/all_wind_heading_and_trajectories.csv')
body_and_trajectory_by_id = [group for _, group in all_fly_data.groupby('trajec_objid')]

temp_augmented_all_fly_data = [augment_fly_trajectory(df, compute_heading_from_ellipses=True) for df in body_and_trajectory_by_id]
augmented_all_fly_data = pd.concat(temp_augmented_all_fly_data, ignore_index=True)
os.makedirs('../pipelinedata/02_augmented', exist_ok=True)
augmented_all_fly_data.to_csv('../pipelinedata/02_augmented/all_wind_heading_and_trajectories_augmented.csv', index=False)

#### Body Orientation Correction and Filtering

The body-orientation measurements were captured with a top-down camera. At each frame, the fly was approximated as an ellipse, so the recorded heading is only accurate modulo π. This produces discontinuous π-jumps throughout many trajectories. Measurement noise also introduces high-frequency jitter.

We apply three sequential steps to correct and smooth the heading signal before use in training.


##### 1. Naïve Heading Correction

A simple heuristic exploiting two behavioral observations: (1) *Drosophila* generally fly with their body axis aligned with the thrust vector, and (2) at the temporal resolution of these measurements (0.01 s), the fly rarely changes heading by more than π/2.

The initial orientation is aligned with the thrust direction, then `np.unwrap` with `period=π` detects and removes subsequent π-flips. This resolves most ambiguities but fails for some dynamic segments — motivating the convex-optimization step below.


In [ ]:
import scipy

from utils import naive_heading_correction

##### 2. Convex-Optimization–Based Heading Correction

To obtain a globally consistent heading signal, we solve a mixed-integer convex program. At each time step $t$, we select $k_t \in \{-1, 0, 1\}$ to add $k_t \pi$ to the raw measurement $\theta_t$, minimizing:

$$\min_{\{k_t\}} \; \alpha_1 \sum_{t=2}^T |\tilde\theta_t - \tilde\theta_{t-1}| + \alpha_2 \sum_{t=1}^T |(\tilde\theta_t - \varphi_t) - \bar\delta|$$

where $\tilde\theta_t = \theta_t + k_t\pi$, $\varphi_t$ is the thrust direction, and $\bar\delta$ is the mean heading–thrust offset. The first term penalizes total variation (smoothness); the second anchors the corrected heading to the thrust direction. Weights $\alpha_1 = 1$, $\alpha_2 = 5$ were tuned on manually inspected trajectories. Solved via `cvxpy` with the MOSEK solver.


In [ ]:
import cvxpy

from utils import convex_opt_heading_correction

##### Visualizing Heading Correction Steps

For troubleshooting and sanity-checking, we visualize a sample of trajectories at each correction stage to confirm the algorithm is behaving as expected.


In [ ]:
import os
import pandas as pd
import datetime
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
augmented_all_fly_data = pd.read_csv('../pipelinedata/02_augmented/all_wind_heading_and_trajectories_augmented.csv')
heading_and_trajectory_by_id = [group for _, group in augmented_all_fly_data.groupby('trajec_objid')]

In [ ]:
from utils import plot_trajectory

In [ ]:
import random
import gc

# Set the random seed for repeatability
random.seed(42)

# Randomly sample 100 trajectories from heading_and_trajectory_by_id
sampled_trajectories = random.sample(heading_and_trajectory_by_id, 100)

# Loop through each trajectory and create/save figures
for trajectory in sampled_trajectories:
    trajec_objid = trajectory["trajec_objid"].iloc[0]

    # Create the subplots with a reduced dpi and adjusted size
    fig, axs = plt.subplots(1, 3, figsize=(10, 10), dpi=100)

    # Plot the original trajectory
    plot_trajectory(axs[0], trajectory)

    # Apply the naive heading correction once, store the result
    naive_correction = naive_heading_correction(trajectory)
    plot_trajectory(axs[1], naive_correction)

    # Apply the convex optimization correction and plot
    convex_correction = convex_opt_heading_correction(naive_correction)
    plot_trajectory(axs[2], convex_correction)

    # Save the figure as an SVG file
    os.makedirs('../pipelinedata/03_corrected/heading_correction_steps_svg', exist_ok=True)
    svg_filename = f'../pipelinedata/03_corrected/heading_correction_steps_svg/{trajec_objid}.svg'
    plt.savefig(svg_filename, format='svg')

    # Close the figure to free up memory
    plt.close(fig)

    # Explicitly call garbage collection to free memory
    gc.collect()

In [ ]:
# plot specific trajectory by id
test_trajec = augmented_all_fly_data.loc[augmented_all_fly_data["trajec_objid"]=="20130325_193727_8614"]
fig, axs = plt.subplots(1, 3, figsize=(12, 12), dpi=150)
plot_trajectory(axs[0],test_trajec)
plot_trajectory(axs[1],naive_heading_correction(test_trajec))
plot_trajectory(axs[2],convex_opt_heading_correction(naive_heading_correction(test_trajec)))

In [ ]:
test_trajec

##### Applying Heading Correction to All Trajectories

We apply both correction algorithms sequentially to every trajectory and save the result.


In [ ]:
import os

augmented_all_fly_data = pd.read_csv('../pipelinedata/02_augmented/all_wind_heading_and_trajectories_augmented.csv')
body_and_trajectory_by_id = [group for _, group in augmented_all_fly_data.groupby('trajec_objid')]
temp_dfs = []
for traj in body_and_trajectory_by_id:
    corrected_trajec = convex_opt_heading_correction(naive_heading_correction(traj))
    temp_dfs.append(corrected_trajec)
corrected_aug_all_fly_data = pd.concat(temp_dfs, ignore_index=True)
os.makedirs('../pipelinedata/03_corrected', exist_ok=True)
corrected_aug_all_fly_data.to_csv('../pipelinedata/03_corrected/all_wind_heading_and_trajectories_augmented_corrected.csv', index=False)

##### Trajectory Filtering

Some trajectories retain residual π-flips that neither correction step fully resolves. We detect these by computing the circular distance between consecutive heading angles and filtering out trajectories that exceed a threshold jump magnitude.


In [ ]:
from utils import circular_distance, calc_circular_difference

In [ ]:
import os

# Load the CSV file
corrected_aug_all_fly_data = pd.read_csv('../pipelinedata/03_corrected/all_wind_heading_and_trajectories_augmented_corrected.csv')

# Group data by 'trajec_objid'
body_and_trajectory_by_id = [group for _, group in corrected_aug_all_fly_data.groupby('trajec_objid')]

# Initialize lists for storing filtered and removed trajectories
temp_dfs = []
temp_bad_dfs = []
removal_count = 0

# Loop through each trajectory and apply filtering based on circular difference
for traj in body_and_trajectory_by_id:
   circ_diff_data = calc_circular_difference(traj)  # Ensure this function returns a DataFrame with 'circ_dist_diff'
   if max(circ_diff_data["circ_dist_diff"]) < (np.pi / 2):  # Filter condition
      temp_dfs.append(traj)  # Keep this trajectory
   else:
      temp_bad_dfs.append(traj)  # Mark this trajectory for removal
      removal_count += 1

# Concatenate the filtered and removed trajectories into separate DataFrames
bad_all_fly_data = pd.concat(temp_bad_dfs, ignore_index=True)
filtered_corr_aug_all_fly_data = pd.concat(temp_dfs, ignore_index=True)

os.makedirs('../pipelinedata/04_filtered', exist_ok=True)

# Save the filtered data to CSV files
filtered_corr_aug_all_fly_data.to_csv('../pipelinedata/04_filtered/all_wind_heading_and_trajectories_augmented_corrected_filtered.csv', index=False)

# Save the rejected trajectories
bad_all_fly_data.to_csv('../pipelinedata/04_filtered/rejected_trajectories.csv', index=False)

# Print the total number of trajectories and the number of removals
print(f"Total trajectories: {len(body_and_trajectory_by_id)}")
print(f"Number of removed trajectories: {removal_count}")

##### Visualizing Filtered-Out Trajectories


In [ ]:
import random
import gc

filtered_corr_aug_all_fly_data = pd.read_csv('../pipelinedata/04_filtered/rejected_trajectories.csv')
body_and_trajectory_by_id = [group for _, group in filtered_corr_aug_all_fly_data.groupby('trajec_objid')]

# Set the random seed for repeatability
random.seed(42)

# Loop through each trajectory and create/save figures
for trajectory in body_and_trajectory_by_id:
    trajec_objid = trajectory["trajec_objid"].iloc[0]
    fig, ax = plt.subplots(figsize=(12, 12), dpi=150)

    plot_trajectory(ax, trajectory, plot_ellipses=False, every_nth=1)

    # Save the figure as an SVG file
    os.makedirs('../pipelinedata/04_filtered/rejected_trajectory_svgs', exist_ok=True)
    svg_filename = f'../pipelinedata/04_filtered/rejected_trajectory_svgs/{trajec_objid}.svg'
    plt.savefig(svg_filename, format='svg')
    plt.close(fig)  # Close the figure to free up memory
    gc.collect()

##### 3. Temporal Smoothing

To reduce measurement jitter, we apply a two-stage procedure. First, the corrected heading is unwrapped to a continuous signal using a sliding-window algorithm that selects the 2π-shifted representative most consistent with recent history. Second, the unwrapped signal is smoothed with a Savitzky–Golay filter (sampling interval 0.01 s) to suppress high-frequency noise while retaining slower turning dynamics. The result is wrapped back to [−π, π] and used as the training target.


This is especially noticeable in downwind trajectories, where the fly's heading can drift slowly relative to its velocity direction.


In [ ]:
from utils import smooth_trajectory

In [ ]:
test_trajec = augmented_all_fly_data.loc[augmented_all_fly_data["trajec_objid"]=="20121002_184808_5511"]
fig, axs = plt.subplots(1, 2, figsize=(12, 12), dpi=150)
plot_trajectory(axs[0],convex_opt_heading_correction(naive_heading_correction(test_trajec)),plot_ellipses=False,every_nth=1,legend=False)
plot_trajectory(axs[1],smooth_trajectory(convex_opt_heading_correction(naive_heading_correction(test_trajec))),plot_ellipses=False,every_nth=1,legend=False)

##### Applying Smoothing to All Trajectories


In [ ]:
import os

augmented_all_fly_data = pd.read_csv('../pipelinedata/04_filtered/all_wind_heading_and_trajectories_augmented_corrected_filtered.csv')
body_and_trajectory_by_id = [group for _, group in augmented_all_fly_data.groupby('trajec_objid')]
temp_dfs = []
test_dfs = []
for traj in body_and_trajectory_by_id:
    test_dfs.append(traj)
    smoothed_trajec = smooth_trajectory(traj)
    temp_dfs.append(smoothed_trajec)
smoothed_corrected_aug_all_fly_data = pd.concat(temp_dfs, ignore_index=True)
os.makedirs('../pipelinedata/05_smoothed', exist_ok=True)
smoothed_corrected_aug_all_fly_data.to_csv('../pipelinedata/05_smoothed/all_wind_heading_and_trajectories_augmented_corrected_filtered_smoothed.csv', index=False)

##### Visualizing Smoothed Trajectories


In [ ]:
filtered_corr_aug_all_fly_data = pd.read_csv('../pipelinedata/05_smoothed/all_wind_heading_and_trajectories_augmented_corrected_filtered_smoothed.csv')
body_and_trajectory_by_id = [group for _, group in filtered_corr_aug_all_fly_data.groupby('trajec_objid')]

In [ ]:
import random
import gc

# Set the random seed for repeatability
random.seed(42)

# Randomly sample 100 trajectories from heading_and_trajectory_by_id
sampled_trajectories = random.sample(body_and_trajectory_by_id, 100)

# Loop through each trajectory and create/save figures
for trajectory in sampled_trajectories:
    trajec_objid = trajectory["trajec_objid"].iloc[0]
    fig, ax = plt.subplots(figsize=(12, 12), dpi=150)

    plot_trajectory(ax, trajectory)

    # Save the figure as an SVG file
    os.makedirs('../pipelinedata/05_smoothed/trajectory_svgs', exist_ok=True)
    svg_filename = f'../pipelinedata/05_smoothed/trajectory_svgs/{trajec_objid}.svg'
    plt.savefig(svg_filename, format='svg')
    plt.close(fig)  # Close the figure to free up memory
    gc.collect()

In [ ]:
# plot specific trajectory by id
test_trajec = filtered_corr_aug_all_fly_data.loc[filtered_corr_aug_all_fly_data["trajec_objid"]=="20121210_201202_2893"]
fig, ax = plt.subplots(figsize=(12, 12), dpi=150)
plot_trajectory(ax,test_trajec)

### Feature Augmentation, Rotation Augmentation, and Delay Embedding

To construct inputs for the neural network, we augment the processed trajectories with a time-delay embedding of six kinematic variables: groundspeed, groundspeed angle, airspeed, airspeed angle, thrust, and thrust angle. A backward-looking window of $W = 4$ timesteps yields $6 \times 4 = 24$ input features per training example. The target output is the heading direction represented as $(\cos\theta_t, \sin\theta_t)$, stored as `heading_angle_x` and `heading_angle_y`.

Note: rotation augmentation (randomly rotating all angular quantities per trajectory to remove wind-direction bias) was applied in the CEM model variant (`model_CEM_all-angle-rotate.keras`) but is not included in the base model trained here.


#### Import Preprocessed Data

Import the fully preprocessed dataset (corrected, filtered, smoothed) and add the `heading_angle_x`/`heading_angle_y` output columns.


In [ ]:
import os
# Import libraries
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np

all_fly_data = pd.read_csv('../pipelinedata/05_smoothed/all_wind_heading_and_trajectories_augmented_corrected_filtered_smoothed.csv')
all_fly_data['heading_angle_x'] = np.cos(all_fly_data['heading_angle']) # augment df with neurel network output parameters (in this case, to avoid circular-fitting issues)
all_fly_data['heading_angle_y'] = np.sin(all_fly_data['heading_angle'])
os.makedirs('../pipelinedata/06_final', exist_ok=True)
all_fly_data.to_csv('../pipelinedata/06_final/all_wind_heading_and_trajectories_augmented_corrected_filtered_headx_heady.csv', index=False)